In [ ]:
# Runnable What & How
# Runnable in langchain is a tool that allows you to define a task (the "What") and the method to accomplish it (the "How"). It provides a structured way to execute tasks using various tools and agents.
# The "What" is the task you want to accomplish, and the "How" is the method or tool you will use to achieve that task. Runnable can be used to create complex workflows by chaining together multiple tasks and methods.
# Runnable can be defined in two ways:
# 1. Task specific: You can define a Runnable for a specific task, like model calling, prompt templates, parsing, retrievers, etc.
# 2. Runnable Primitives: specific orchestration tools used to define the logic, data flow, and control structures of your chains. These primitives include:
# Basically these premitives allowes to Run Runnable Tasks in different ways, such as sequentially, in parallel, with branching logic, with retries, with timeouts, with caching, etc. They provide the building blocks for creating complex workflows and orchestrating the execution of tasks in a flexible and efficient manner.
# - `RunnableLambda`: A simple wrapper around a Python function(allows to make any function as Runnable), allowing you to define custom logic in a Runnable.
# - `RunnableSequence`: A tool for executing a sequence of Runnables in order, passing the output of one as the input to the next.
# - `RunnableParallel`: A tool for executing multiple Runnables in parallel, allowing you to perform multiple tasks simultaneously and aggregate their results.
# - `RunnableBranch`: A tool for branching logic based on conditions(used for conditional chain execution), allowing you to execute different Runnables based on the input.
# - `RunnablePassthrough`: A tool for passing data through a Runnable without modification, allowing you to include data in the workflow without processing it.
# - `RunnableMap`: A tool for applying a Runnable to each item in a list, allowing you to process multiple items in parallel.
# - `RunnableReduce`: A tool for reducing a list of items into a single output using a specified function, allowing you to aggregate results from multiple Runnables.
# - `RunnableFilter`: A tool for filtering a list of items based on a specified condition, allowing you to select only the items that meet certain criteria.
# - `RunnableRetry`: A tool for retrying a Runnable a specified number of times in case of failure, allowing you to handle transient errors and improve the robustness of your workflows.
# - `RunnableTimeout`: A tool for setting a timeout on a Runnable, allowing you to limit the execution time of a task and handle cases where it takes too long to complete.
# - `RunnableCache`: A tool for caching the results of a Runnable, allowing you to avoid redundant computations and improve performance by reusing previously computed results.
# - `RunnableLog`: A tool for logging the execution of a Runnable, allowing you to track the progress and results of your workflows for debugging and monitoring purposes.  

# etc.



#Note: As RunnableSequence is mostly used Runnable primitive,
#      so, langchain team provided shorthand for it,
#      instead of Writing RunnableSequece we can use | operator to chain Runnables together.
#      This is alse called LCEL (Langchain Expression Language) which provides a more concise and readable way to define chains of Runnables.

In [4]:
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate

# Load environment variables from .env file
load_dotenv()

parser = StrOutputParser()

In [5]:
#Model loading
model = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite", temperature=0, max_tokens=2048)

In [6]:
# RunnableSequence Example
from langchain_core.runnables import RunnableSequence

prompt1 = PromptTemplate(
    template='Write a joke about {topic}',
    input_variables=['topic']
)

prompt2 = PromptTemplate(
    template='Explain the following joke - {text}',
    input_variables=['text']
)

chain = RunnableSequence(prompt1, model, parser, prompt2, model, parser)
result = chain.invoke({'topic': 'UPSC'})

In [7]:
print(result)

This joke plays on a common idiom and a relatable situation for UPSC aspirants. Here's the breakdown:

*   **UPSC Aspirant:** This refers to someone preparing for the Union Public Service Commission (UPSC) exams in India. These exams are notoriously difficult and highly competitive, with a very low pass rate.

*   **Cut-off Marks:** In competitive exams, the "cut-off marks" are the minimum scores required to pass or to be selected for the next stage of the process.

*   **"Sky-high":** This is an idiom that means something is extremely high, often to an unreasonable or unattainable degree.

**The Joke's Logic:**

1.  **The Literal Interpretation:** The aspirant brings a ladder. A ladder is used to reach things that are physically high up.

2.  **The Figurative Interpretation:** The aspirant *hears* that the cut-off marks are "sky-high." This means they believe the required scores are incredibly high and difficult to achieve.

3.  **The Punchline:** The aspirant, taking the idiom litera

In [8]:
# RunnableLambda Example
from langchain_core.runnables import RunnableLambda
def add_exclamation(text: str) -> str:
    return text + "!"
exclaim = RunnableLambda(add_exclamation)
result = exclaim.invoke("Hello")
print(result)

Hello!


In [12]:
#Runnable Paralllel Example
from langchain_core.runnables import RunnableParallel

prompt1 = PromptTemplate(
    template='Generate a tweet about {topic} in less than 20 words.',
    input_variables=['topic']
)

prompt2 = PromptTemplate(
    template='Generate a LinkedIn post about {topic} in less than 50 words.',
    input_variables=['topic']
)

parallel_chain = RunnableParallel({
    # 'tweet': RunnableSequence(prompt1, model, parser),
    # 'linkedin': RunnableSequence(prompt2, model, parser)
    'tweet': prompt1 | model | parser,
    'linkedin': prompt2 | model | parser
})

result = parallel_chain.invoke({'topic':'AI'})
print(result)

{'tweet': 'AI: Transforming our world, one algorithm at a time. #AI #Future', 'linkedin': "AI is transforming industries at lightning speed! From automating tasks to unlocking new insights, its potential is immense. What's the most exciting AI application you've seen recently? #AI #Innovation #FutureOfWork"}


In [ ]:
# Runnable Passthrough Example
from langchain_core.runnables import RunnablePassthrough

prompt1 = PromptTemplate(
    template='Write a laughing joke about {topic}',
    input_variables=['topic']
)

prompt2 = PromptTemplate(
    template='Explain the following joke - {text} in less than 200 words.',
    input_variables=['text']
)

joke_gen_chain = RunnableSequence(prompt1, model, parser)

parallel_chain =RunnableParallel({
    'joke': RunnablePassthrough(),
    'explanation': RunnableSequence(prompt2, model, parser)
})

final_chain = RunnableSequence(joke_gen_chain, parallel_chain)

result = final_chain.invoke({'topic': 'UPSC'})
print(result)

# if you directly pass joke_gen_chain inside parallel chain itself, then it will go to explaination chain only, 
# and joke will not be generated as joke_gen_chain is not directly connected to joke key in parallel chain, 
# so we need to use RunnablePassthrough to pass the output of joke_gen_chain to parallel chain, and 
# then it will be available for explaination chain to use it.

{'joke': 'Why did the UPSC aspirant bring a ladder to the exam hall?\n\nBecause they heard the cut-off marks were *sky-high*!', 'explanation': 'This joke plays on a common idiom and the reality of competitive exams.\n\n"Sky-high" is an expression meaning extremely high. In the context of the UPSC (Union Public Service Commission) exams, the "cut-off marks" are the minimum scores required to pass or qualify for the next stage. These cut-offs are notoriously high due to the intense competition.\n\nThe humor comes from the literal interpretation of "sky-high." The UPSC aspirant, hearing the cut-off is "sky-high," humorously brings a ladder, as if to physically climb to reach those elevated marks. It\'s a silly, visual gag that highlights the daunting nature of achieving the required score.'}


In [17]:
# RunnableBranch Example
from langchain_core.runnables import RunnableBranch

prompt1 = PromptTemplate(
    template='Write a comprehensive report on {topic}',
    input_variables=['topic']
)

prompt2 = PromptTemplate(
    template='Write a short summary on {text} in less than 50 words.',
    input_variables=['text']
)

comprehensive_chain = RunnableSequence(prompt1, model, parser)

branch_chain = RunnableBranch(
    (lambda x: len(x.split()) > 100,prompt2 | model | parser), # if comprehensive report is more than 100 words, then we will generate summary, otherwise we will return comprehensive report only.
    RunnablePassthrough()
)

final_chain = RunnableSequence(comprehensive_chain, branch_chain)
result = final_chain.invoke({'topic': 'AI'})
print(result)


This report provides a comprehensive overview of Artificial Intelligence (AI), a field focused on creating intelligent machines. It covers AI's history, core concepts like machine learning and deep learning, and its diverse applications across sectors such as healthcare, finance, and transportation. The report highlights AI's current state, dominated by ML/DL and generative AI, while also addressing critical ethical challenges like bias, privacy, and job displacement.
